# Berlin – Noise Data Analysis

This notebook visualises the Berlin noise exposure dataset (`Berlin_noise_points.gpkg`).  
The dataset contains **~3.8 M building-facade noise points** with Lden / Lnight values  
broken down by source: road traffic (`STR`), rail (`SCH`), industrial (`IED`), air traffic (`FLG`)  
and total combined (`GES`).  
CRS: **EPSG:25833** (ETRS89 / UTM zone 33N).  
OSM street network is downloaded and used as a background layer for all spatial plots.

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import osmnx as ox

## Load noise data

In [ ]:
_p = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'layers', 'Berlin_noise_points.gpkg')
print('Loading from:', _p)

noise = gpd.read_file(_p)
print('CRS:', noise.crs)
print('Shape:', noise.shape)
print('Columns:', noise.columns.tolist())

## Explore columns

In [ ]:
print('dtypes:')
print(noise.dtypes)
print()
print('NUTZUNG (building type):')
print(noise['NUTZUNG'].value_counts())
print()
print('RICHTUNG (facade direction):')
print(noise['RICHTUNG'].value_counts())
print()

noise_cols = ['STR_DEN', 'STR_N', 'SCH_DEN', 'SCH_N',
              'IED_DEN', 'IED_N', 'FLG_DEN', 'FLG_N',
              'AEG_DEN', 'AEG_N', 'GES_DEN', 'GES_N']
print('Noise statistics:')
print(noise[noise_cols].describe().round(1))

## Subsample for visualisation

The full dataset (3.8 M points) is too large for interactive / static plotting.  
We take a random stratified subsample of **100 000 points**, preserving the  
building-type distribution.

In [ ]:
N_SAMPLE = 100_000

parts = []
for nutzung, grp in noise.groupby('NUTZUNG'):
    n = max(1, round(N_SAMPLE * len(grp) / len(noise)))
    parts.append(grp.sample(n=min(n, len(grp)), random_state=42))

noise_sub = gpd.GeoDataFrame(pd.concat(parts), crs=noise.crs)
if len(noise_sub) > N_SAMPLE:
    noise_sub = noise_sub.sample(n=N_SAMPLE, random_state=42)

print(f'Subsample size: {len(noise_sub)}')
print(noise_sub['NUTZUNG'].value_counts())

## Add noise class bins

Following the EU END classification: < 45, 45–50, 50–55, 55–60, 60–65, 65–70, ≥ 70 dB(A).

In [ ]:
BINS   = [0, 45, 50, 55, 60, 65, 70, 200]
LABELS = ['< 45', '45–50', '50–55', '55–60', '60–65', '65–70', '≥ 70']

noise_sub['ges_den_class'] = pd.cut(
    noise_sub['GES_DEN'], bins=BINS, labels=LABELS, right=False
)

# Also classify the full dataset for distribution charts
noise['ges_den_class'] = pd.cut(
    noise['GES_DEN'], bins=BINS, labels=LABELS, right=False
)

print('Full dataset – point count by noise class:')
print(noise['ges_den_class'].value_counts().sort_index())

## Download OSM street network

Streets are fetched for the Berlin area and used as a background for all spatial plots.

In [ ]:
GRAPHML_PATH = os.path.join('data', 'berlin.graphml')
os.makedirs('data', exist_ok=True)

if os.path.exists(GRAPHML_PATH):
    print('Loading cached graph from', GRAPHML_PATH)
    G = ox.load_graphml(GRAPHML_PATH)
else:
    print('Downloading Berlin OSM graph (drive network) ...')
    G = ox.graph_from_place('Berlin, Germany', network_type='drive')
    ox.save_graphml(G, GRAPHML_PATH)
    print('Saved to', GRAPHML_PATH)

_, streets = ox.graph_to_gdfs(G)
streets = streets[streets.geometry.type == 'LineString'].to_crs(noise_sub.crs)
print('Streets loaded:', len(streets), 'segments, CRS:', streets.crs)

## Static map – total noise Lden (GES_DEN)

Street network in light grey, noise points coloured by total combined Lden.

In [ ]:
cmap = plt.cm.coolwarm
norm = mcolors.Normalize(vmin=40, vmax=75)

fig, ax = plt.subplots(figsize=(14, 14))

# Background streets
streets.plot(ax=ax, color='#cccccc', linewidth=0.3, zorder=1)

# Noise points
noise_sub.plot(
    ax=ax,
    column='GES_DEN',
    cmap=cmap,
    norm=norm,
    markersize=0.5,
    alpha=0.6,
    legend=True,
    legend_kwds={'label': 'Total Lden dB(A)', 'orientation': 'vertical', 'shrink': 0.6},
    zorder=2
)

ax.set_title('Berlin – Building Facade Noise Exposure (Total Lden)', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Static map – road traffic noise Lden (STR_DEN)

In [ ]:
str_sub = noise_sub.dropna(subset=['STR_DEN'])

fig, ax = plt.subplots(figsize=(14, 14))
streets.plot(ax=ax, color='#cccccc', linewidth=0.3, zorder=1)
str_sub.plot(
    ax=ax,
    column='STR_DEN',
    cmap=cmap,
    norm=norm,
    markersize=0.5,
    alpha=0.6,
    legend=True,
    legend_kwds={'label': 'Road Traffic Lden dB(A)', 'orientation': 'vertical', 'shrink': 0.6},
    zorder=2
)
ax.set_title('Berlin – Road Traffic Noise Exposure (STR Lden)', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Static map – total noise Lnight (GES_N)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))
streets.plot(ax=ax, color='#cccccc', linewidth=0.3, zorder=1)
noise_sub.plot(
    ax=ax,
    column='GES_N',
    cmap=cmap,
    norm=mcolors.Normalize(vmin=30, vmax=65),
    markersize=0.5,
    alpha=0.6,
    legend=True,
    legend_kwds={'label': 'Total Lnight dB(A)', 'orientation': 'vertical', 'shrink': 0.6},
    zorder=2
)
ax.set_title('Berlin – Building Facade Noise Exposure (Total Lnight)', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Static map – noise by class (GES_DEN binned)

In [ ]:
CLASS_COLORS = {
    '< 45':   '#2166ac',
    '45–50':  '#74add1',
    '50–55':  '#abd9e9',
    '55–60':  '#fee090',
    '60–65':  '#f46d43',
    '65–70':  '#d73027',
    '≥ 70':   '#a50026',
}

fig, ax = plt.subplots(figsize=(14, 14))
streets.plot(ax=ax, color='#cccccc', linewidth=0.3, zorder=1)

for cls, color in CLASS_COLORS.items():
    subset = noise_sub[noise_sub['ges_den_class'] == cls]
    if len(subset):
        subset.plot(ax=ax, color=color, markersize=0.5, alpha=0.6, zorder=2, label=cls)

handles = [mpatches.Patch(color=c, label=l) for l, c in CLASS_COLORS.items()]
ax.legend(handles=handles, title='Lden dB(A)', loc='lower right', fontsize=10)
ax.set_title('Berlin – Noise Exposure Classes (Total Lden)', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Interactive map

In [ ]:
# Use a smaller subsample for the interactive map to keep it responsive
interactive_sub = noise_sub.sample(n=min(20_000, len(noise_sub)), random_state=42).to_crs('EPSG:4326')

interactive_sub.explore(
    column='GES_DEN',
    cmap='coolwarm',
    vmin=40, vmax=75,
    marker_kwds={'radius': 3},
    tooltip=['NUTZUNG', 'RICHTUNG', 'GES_DEN', 'GES_N', 'STR_DEN', 'SCH_DEN'],
    popup=True,
    tiles='CartoDB positron',
    legend_kwds={'caption': 'Total Lden dB(A)'},
)

## Point count by noise class

In [ ]:
class_counts = noise['ges_den_class'].value_counts().reindex(LABELS).fillna(0)
class_colors = list(CLASS_COLORS.values())

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(class_counts.index, class_counts.values / 1000,
       color=class_colors, edgecolor='white', linewidth=0.5)
ax.set_title('Point Count by Noise Class (Total Lden)', fontsize=14, fontweight='bold')
ax.set_xlabel('Noise Range dB(A)')
ax.set_ylabel('Number of Points (thousands)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(class_counts.to_string())

## Cumulative distribution

In [ ]:
total_pts = class_counts.sum()
class_pct = class_counts / total_pts * 100
cumulative_pct = class_pct.cumsum()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(class_pct.index, class_pct.values,
        color=class_colors, edgecolor='white', linewidth=0.5, label='Share %')
ax1.set_ylabel('Share of facade points (%)')
ax1.set_xlabel('Noise Range dB(A)')
ax1.set_title('Distribution and Cumulative Coverage of Facade Points', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(cumulative_pct.index, cumulative_pct.values,
         color='black', marker='o', linewidth=2, label='Cumulative %')
ax2.set_ylabel('Cumulative share (%)')
ax2.set_ylim(0, 105)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

## Noise by building type (NUTZUNG)

In [ ]:
nutzung_stats = (
    noise.groupby('NUTZUNG')[['GES_DEN', 'GES_N', 'STR_DEN']]
    .agg(['mean', 'median', 'std'])
    .round(1)
)
print(nutzung_stats.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
nutzung_groups = noise.groupby('NUTZUNG')['GES_DEN'].mean().sort_values(ascending=False)
colors = plt.cm.tab10.colors[:len(nutzung_groups)]
ax.bar(nutzung_groups.index, nutzung_groups.values, color=colors, edgecolor='white')
ax.set_title('Mean Total Lden by Building Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean GES_DEN dB(A)')
ax.set_xlabel('Building Type (NUTZUNG)')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## Noise source breakdown

In [ ]:
source_cols = ['STR_DEN', 'SCH_DEN', 'IED_DEN', 'FLG_DEN']
source_labels = ['Road Traffic', 'Rail', 'Industrial', 'Air Traffic']
source_colors = ['#e6194B', '#3cb44b', '#4363d8', '#f58231']

source_means = noise[source_cols].mean(skipna=True)
source_coverage = noise[source_cols].notna().sum() / len(noise) * 100

print('Mean Lden by source (where available):')
for col, label in zip(source_cols, source_labels):
    print(f'  {label}: {source_means[col]:.1f} dB(A)  '
          f'(covers {source_coverage[col]:.1f}% of points)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(source_labels, source_means.values, color=source_colors, edgecolor='white')
ax1.set_title('Mean Lden by Noise Source', fontsize=13, fontweight='bold')
ax1.set_ylabel('Mean dB(A)')
ax1.grid(axis='y', alpha=0.3)

ax2.bar(source_labels, source_coverage.values, color=source_colors, edgecolor='white')
ax2.set_title('Coverage (% of facade points affected)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Coverage (%)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Noise source comparison maps

Side-by-side: road traffic vs rail vs air traffic Lden on the street background.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

plot_sources = [
    ('STR_DEN', 'Road Traffic Lden'),
    ('SCH_DEN', 'Rail Lden'),
    ('FLG_DEN', 'Air Traffic Lden'),
]

for ax, (col, title) in zip(axes, plot_sources):
    sub = noise_sub.dropna(subset=[col])
    streets.plot(ax=ax, color='#cccccc', linewidth=0.3, zorder=1)
    sub.plot(
        ax=ax,
        column=col,
        cmap=cmap,
        norm=mcolors.Normalize(vmin=40, vmax=75),
        markersize=0.5,
        alpha=0.7,
        legend=False,
        zorder=2
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_axis_off()

sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(vmin=40, vmax=75))
sm.set_array([])
fig.colorbar(sm, ax=axes, orientation='vertical', fraction=0.015, pad=0.02,
             label='Lden dB(A)')

fig.suptitle('Berlin – Noise Exposure by Source', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
total_pts = len(noise)
mean_ges_den = noise['GES_DEN'].mean()
mean_ges_n   = noise['GES_N'].mean()
mean_str_den = noise['STR_DEN'].mean(skipna=True)
dominant_cls = noise['ges_den_class'].value_counts().idxmax()

print(f'Total facade points  : {total_pts:,}')
print(f'Mean total Lden      : {mean_ges_den:.1f} dB(A)')
print(f'Mean total Lnight    : {mean_ges_n:.1f} dB(A)')
print(f'Mean road Lden       : {mean_str_den:.1f} dB(A)')
print(f'Dominant noise class : {dominant_cls}')
print()
print('Points above 55 dB(A) Lden (EU health threshold):')
above_55 = (noise['GES_DEN'] >= 55).sum()
print(f'  {above_55:,} ({above_55/total_pts*100:.1f}%)')
print()
print('Points above 65 dB(A) Lden (EU high exposure threshold):')
above_65 = (noise['GES_DEN'] >= 65).sum()
print(f'  {above_65:,} ({above_65/total_pts*100:.1f}%)')
print()
print('Count distribution by noise class:')
print(noise['ges_den_class'].value_counts().reindex(LABELS).fillna(0).astype(int).to_string())